In [1]:
import os
os.getcwd()

'/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant/notebooks'

In [2]:
os.chdir('/Users/komalpreet/Desktop/Github/Amazon_Product_Query_Assistant/')

In [5]:
from datasets import load_dataset

reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    split="full",
    trust_remote_code=True,
)

meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    split="full",
    trust_remote_code=True,
)

print(reviews[0])
print(meta[0])

{'rating': 5.0, 'title': 'Such a lovely scent but not overpowering.', 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!", 'images': [], 'asin': 'B00YQ6X8EO', 'parent_asin': 'B00YQ6X8EO', 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ', 'timestamp': 1588687728923, 'helpful_vote': 0, 'verified_purchase': True}
{'main_category': 'All Beauty', 'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)', 'average_rating': 4.8, 'rating_number': 10, 'features': [], 'description': [], 'price': 'None', 'images': {'hi_res': [None, 'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'], 'large': ['https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg', 'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg'], 'thumb': ['https://m.media-ama

In [6]:
# how many records?
print("Reviews:", len(reviews))
print("Meta:", len(meta))

# check fields
print("\nReview fields:", reviews[0].keys())
print("Meta fields:", meta[0].keys())

Reviews: 701528
Meta: 112590

Review fields: dict_keys(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase'])
Meta fields: dict_keys(['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'])


In [7]:
import pandas as pd

df_meta = pd.DataFrame(meta)
print(df_meta[['title', 'price', 'average_rating', 'rating_number', 'store', 'features', 'description']].isnull().sum())
print("\nPrice sample:", df_meta['price'].dropna().head())

title                 0
price                 0
average_rating        0
rating_number         0
store             11331
features              0
description           0
dtype: int64

Price sample: 0    None
1    None
2    None
3    None
4    None
Name: price, dtype: str


In [8]:
# check price more carefully
print("Unique price samples:", df_meta['price'].unique()[:10])

# check features and description — are they actually populated?
print("\nEmpty features:", df_meta['features'].apply(lambda x: len(x) == 0).sum())
print("Empty description:", df_meta['description'].apply(lambda x: len(x) == 0).sum())

# check details
print("\nDetails sample:", df_meta['details'].dropna().iloc[0])

Unique price samples: <ArrowStringArray>
[ 'None',  '6.99', '86.95',  '79.5',  '5.99',  '29.8',  '24.0', '22.49',
 '11.99',  '50.0']
Length: 10, dtype: str

Empty features: 95213
Empty description: 93428

Details sample: {"Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds", "UPC": "617390882781"}


In [9]:
df_reviews = pd.DataFrame(reviews)

# how many reviews per product?
reviews_per_product = df_reviews.groupby('parent_asin').size()
print(reviews_per_product.describe())

# how many products have at least 1 review?
print("\nProducts with reviews:", reviews_per_product.shape[0])

# text quality
print("\nEmpty review text:", df_reviews['text'].apply(lambda x: len(str(x).strip()) == 0).sum())
print("Verified purchase %:", df_reviews['verified_purchase'].mean() * 100)

count    112565.000000
mean          6.232204
std          25.189840
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max        1962.000000
dtype: float64

Products with reviews: 112565

Empty review text: 720
Verified purchase %: 90.51228176209645


In [10]:
import ast

# how populated are details?
print("Empty details:", df_meta['details'].apply(lambda x: x == '{}' or x == '' or x is None).sum())

# parse and check keys
from collections import Counter
all_keys = Counter()
for d in df_meta['details']:
    try:
        parsed = ast.literal_eval(d)
        if parsed:
            all_keys.update(parsed.keys())
    except:
        continue

print("\nTop 10 detail keys:", all_keys.most_common(10))

Empty details: 4512

Top 10 detail keys: [('Brand', 72113), ('Package Dimensions', 67825), ('UPC', 60252), ('Is Discontinued By Manufacturer', 39527), ('Item Form', 32439), ('Material', 30942), ('Hair Type', 26719), ('Unit Count', 23855), ('Product Dimensions', 23277), ('Age Range (Description)', 22929)]


In [11]:
import json
from pathlib import Path

Path("data/raw").mkdir(parents=True, exist_ok=True)

# save raw reviews
with open("data/raw/reviews_raw.jsonl", "w") as f:
    for row in reviews:
        f.write(json.dumps(dict(row)) + "\n")

# save raw meta
with open("data/raw/meta_raw.jsonl", "w") as f:
    for row in meta:
        f.write(json.dumps(dict(row)) + "\n")

print("Done!")

Done!


In [12]:
import os

reviews_size = os.path.getsize("data/raw/reviews_raw.jsonl")
meta_size    = os.path.getsize("data/raw/meta_raw.jsonl")

print(f"Reviews: {reviews_size / 1024 / 1024:.1f} MB")
print(f"Meta:    {meta_size / 1024 / 1024:.1f} MB")

Reviews: 311.5 MB
Meta:    195.9 MB


In [13]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

import sys
sys.path.append(".")

from src.preprocessor import build_products

products = build_products(meta, reviews)
print(f"Total products: {len(products)}")
print(f"\nSample search_text:\n{products[0]['search_text']}")

2026-05-03 13:22:58,449 Grouping reviews by parent_asin...
2026-05-03 13:23:09,600   112,565 unique products have reviews
2026-05-03 13:23:09,600 Building product documents...
2026-05-03 13:23:14,616 Total products built: 112,590
2026-05-03 13:23:15,405 Saved → data/processed/products.jsonl


Total products: 112590

Sample search_text:
Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack) | Store: Howard Products | Reviews: Absolutely fabulous - I will never use anything else. This is amazing.  I bought one at a local shop....I then looked at Amazon and purchased 5 more.  My beautiful leather sofa and chairs now look like new.  You will not believe it. Of all leather products I have used this one seems to be the best. I would recommend this product as the last coat for leather (if you have really old dried leather I would recommend covering it with neats foot oil first) the wax in this product will build up some wa Product works as advertised.. I have a 10 year old Bernhart leather chair and ottoman that has been in a room with a fireplace it's entire life. The leather has become very dry as we mostly used spray on leather conditioners. I bo


In [14]:
import pandas as pd
df_meta = pd.DataFrame(meta)
print(df_meta['price'].value_counts().head(10))
print("\nNull/None count:", df_meta['price'].apply(lambda x: x is None or str(x).strip().lower() in ('none', '', 'null')).sum())

price
None     94886
9.99       631
19.99      363
14.99      314
8.99       283
7.99       280
6.99       275
11.99      252
12.99      252
5.99       243
Name: count, dtype: int64

Null/None count: 94886


In [15]:
# check search_text length
import numpy as np
lengths = [len(p['search_text'].split()) for p in products]
print(f"Avg words: {np.mean(lengths):.0f}")
print(f"Max words: {np.max(lengths)}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")

Avg words: 108
Max words: 2305
95th percentile: 299


In [16]:
print(products[0]['top_reviews'])

[{'title': 'Absolutely fabulous - I will never use anything else', 'text': 'This is amazing.  I bought one at a local shop....I then looked at Amazon and purchased 5 more.  My beautiful leather sofa and chairs now look like new.  You will not believe it.', 'rating': 5.0}, {'title': 'Of all leather products I have used this one seems to be the best', 'text': 'I would recommend this product as the last coat for leather (if you have really old dried leather I would recommend covering it with neats foot oil first) the wax in this product will build up some water resistance as a last coat.  But if you want to rejuvenate and protect recent leather I have not found a product any better than this one.', 'rating': 5.0}, {'title': 'Product works as advertised.', 'text': "I have a 10 year old Bernhart leather chair and ottoman that has been in a room with a fireplace it's entire life. The leather has become very dry as we mostly used spray on leather conditioners. I bought 4 bottles of this cream

In [17]:
import importlib
import sys

# remove cached module
if 'src.preprocessor' in sys.modules:
    del sys.modules['src.preprocessor']

from src.preprocessor import build_products
products = build_products(meta, reviews)

import numpy as np
lengths = [len(p['search_text'].split()) for p in products]
print(f"Avg words: {np.mean(lengths):.0f}")
print(f"Max words: {np.max(lengths)}")
print(f"95th percentile: {np.percentile(lengths, 95):.0f}")

2026-05-03 13:23:20,651 Grouping reviews by parent_asin...
2026-05-03 13:23:31,572   112,565 unique products have reviews
2026-05-03 13:23:31,573 Building product documents...
2026-05-03 13:23:37,206 Total products built: 112,590
2026-05-03 13:23:38,100 Saved → data/processed/products.jsonl


Avg words: 108
Max words: 2305
95th percentile: 299


In [18]:
import sys
sys.path.append(".")

import nltk
nltk.download("stopwords")

from src.utils import tokenize, build_corpus

# test tokenizer
print(tokenize("Best moisturizer for sensitive skin!"))

# build corpus
corpus, tokenized_corpus = build_corpus(products)
print(f"Corpus size: {len(corpus)}")
print(f"Sample tokens: {tokenized_corpus[0][:10]}")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/komalpreet/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
2026-05-03 13:23:39,057 Building corpus from 112,590 products...


['best', 'moisturizer', 'sensitive', 'skin']


2026-05-03 13:23:41,426 Corpus built: 112,590 documents


Corpus size: 112590
Sample tokens: ['howard', 'lc0008', 'leather', 'conditioner', '8ounce', '4pack', 'store', 'howard', 'products', 'reviews']


In [19]:
import sys

# remove cached module
if 'src.utils' in sys.modules:
    del sys.modules['src.utils']

from src.utils import tokenize, build_corpus

# test tokenizer
print(tokenize("Best moisturizer for sensitive skin!"))

# build corpus
corpus, tokenized_corpus = build_corpus(products)
print(f"Corpus size: {len(corpus)}")
print(f"Sample tokens: {tokenized_corpus[0][:10]}")

2026-05-03 13:23:41,430 Building corpus from 112,590 products...


['best', 'moisturizer', 'sensitive', 'skin']


2026-05-03 13:23:43,158 Corpus built: 112,590 documents


Corpus size: 112590
Sample tokens: ['howard', 'lc0008', 'leather', 'conditioner', '8ounce', '4pack', 'store', 'howard', 'products', 'reviews']


In [20]:
import sys
!{sys.executable} -m pip install rank_bm25

In [22]:
from src.bm25 import build_bm25, search_bm25, load_bm25
bm25 = build_bm25(tokenized_corpus)

results = search_bm25(bm25, products, "moisturizer for sensitive skin", top_k=5)
for r in results:
    print(f"{r['bm25_score']:.4f} | {r['title']}")

2026-05-03 13:24:42,776 Building BM25 index over 112,590 documents...
2026-05-03 13:24:44,444 BM25 index saved → data/processed/bm25_index.pkl
2026-05-03 13:24:45,157 Tokenized corpus saved → data/processed/tokenized_corpus.pkl


16.8432 | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pack of 3)
16.0985 | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive Skin With Broad Spectrum Spf 15, 4 oz (Pack of 3)
15.4566 | Revitalizing Light-Weight Moisturizer SPF 15
14.9437 | Simple Hydrating Light Moisturizer, 4.2 Ounce 6-pack
14.5508 | BRTC Perfect Calming Cream 50ml, for Dry, Sensitive and Itchy Skin


In [23]:
results = search_bm25(bm25, products, "best shampoo for curly hair", top_k=5)
for r in results:
    print(f"{r['bm25_score']:.4f} | {r['title']}")

13.0654 | 2 pck of Hotheads Clean Shampoo 8 oz
12.2414 | Shea Moisture Jamaican Black Shampoo 13 Ounce (384ml) (Pack of 3)
12.2097 | J.R. Liggett's, Old Fashioned Bar, Shampoo, Jojoba & Peppermint, 3.5 oz (99 g) - 2pc
11.9237 | NORMADENSE 1 Prowash Thickening Shampoo. Normalizing Thickening Shampoo | Biotin Shampoo for, Dry, Weakened, Normal to Thin-Looking Hair | Vegan Hair Shampoo
11.8216 | Lee Stafford Bigger Fatter Fuller Volumizing Shampoo - For limp and fine hair


In [24]:
import sys
if 'src.semantic' in sys.modules:
    del sys.modules['src.semantic']

from src.semantic import build_semantic_index, search_semantic
from sentence_transformers import SentenceTransformer

# build index (will take 5-10 mins on CPU)
index, embeddings = build_semantic_index(corpus)

2026-05-03 13:24:55,862 Loading model: all-MiniLM-L6-v2
2026-05-03 13:24:55,893 Use pytorch device_name: mps
2026-05-03 13:24:55,893 Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/Users/komalpreet/miniconda3/envs/inflection/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2026-05-03 13:24:57,361 Embedding 112,590 documents (batch_size=256)...
Batches: 100%|████████████████████████████████████████████████████████████████████████████| 440/440 [15:40<00:00,  2.14s/it]
2026-05-03 13:40:43,352 Embeddings shape: (112590, 384)
2026-05-03 13:40:43,426 FAISS index built: 112590 vectors
2026-05-03 13:40:43,794 FAISS index saved → data/processed/faiss.index
2026-05-03 13:40:43,797 Embeddings saved  → data/processed/embeddings.npy


In [25]:
from src.semantic import search_semantic, load_semantic_index
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
index, embeddings = load_semantic_index()

results = search_semantic(index, products, "moisturizer for sensitive skin", top_k=5, model=model)
for r in results:
    print(f"{r['semantic_score']:.4f} | {r['title']}")

2026-05-03 13:41:37,669 Use pytorch device_name: mps
2026-05-03 13:41:37,670 Load pretrained SentenceTransformer: all-MiniLM-L6-v2
2026-05-03 13:41:45,567 Loading FAISS index from data/processed/faiss.index...
2026-05-03 13:41:45,835 FAISS index loaded: 112590 vectors
Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.45s/it]

0.7514 | Senka Perfect emulsion Silky Moisture moisturizing lotion 150ml
0.7011 | Living Nature Sensitive Skin Day Moisture Cream
0.6976 | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For Daily Use With Dispensing Pump - With Aloe Vera Jojoba Oil Shea Butter Green Tea For Sensitive Dry And Oily Skin - Premium Nature
0.6934 | Moisturel Therapeutic Lotion 14 oz (Pack of 3)
0.6926 | Cetaphil Moisturizing Cream 20 oz, 2 Pack


In [26]:
query = "moisturizer for sensitive skin"

print("BM25 Results:")
bm25_results = search_bm25(bm25, products, query, top_k=5)
for r in bm25_results:
    print(f"  {r['bm25_score']:.4f} | {r['title']}")

print("\nSemantic Results:")
semantic_results = search_semantic(index, products, query, top_k=5, model=model)
for r in semantic_results:
    print(f"  {r['semantic_score']:.4f} | {r['title']}")

BM25 Results:
  16.8432 | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pack of 3)
  16.0985 | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive Skin With Broad Spectrum Spf 15, 4 oz (Pack of 3)
  15.4566 | Revitalizing Light-Weight Moisturizer SPF 15
  14.9437 | Simple Hydrating Light Moisturizer, 4.2 Ounce 6-pack
  14.5508 | BRTC Perfect Calming Cream 50ml, for Dry, Sensitive and Itchy Skin

Semantic Results:


Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:07<00:00,  7.91s/it]

  0.7514 | Senka Perfect emulsion Silky Moisture moisturizing lotion 150ml
  0.7011 | Living Nature Sensitive Skin Day Moisture Cream
  0.6976 | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For Daily Use With Dispensing Pump - With Aloe Vera Jojoba Oil Shea Butter Green Tea For Sensitive Dry And Oily Skin - Premium Nature
  0.6934 | Moisturel Therapeutic Lotion 14 oz (Pack of 3)
  0.6926 | Cetaphil Moisturizing Cream 20 oz, 2 Pack


In [27]:
query = "something gentle for my skin"

print("BM25 Results:")
bm25_results = search_bm25(bm25, products, query, top_k=5)
for r in bm25_results:
    print(f"  {r['bm25_score']:.4f} | {r['title']}")

print("\nSemantic Results:")
semantic_results = search_semantic(index, products, query, top_k=5, model=model)
for r in semantic_results:
    print(f"  {r['semantic_score']:.4f} | {r['title']}")

BM25 Results:
  12.5509 | Innerest Facial Sheet Mask Colostrum Ampoule Mist Natural K-Beauty (Fragrance Free, 1 Step 10pk Colostrum(0.84fl oz))
  12.2621 | Essano Gentle Facial Cleansing Micellar Wipes, 20 sheets
  12.2043 | Bath Brush Bamyko Silicone Shower Loofah Brush 2 in 1 Face & Body Gentle Scrub Skin Exfoliation Massage Nubs for Baby, Men and Women
  11.7282 | Senzokan - Famous Traditional Beauty Herbal Soap - Deeply Cleanses and Gentle Care your Skin with the Medical Plants (1 x 100 gr)
  11.6467 | Elite - Retinol Serum 2.5% - | Contains Hyaluronic Acid, Vitamin E, Aloe, Green Tea Extract | Manage Blemish Prone Skin | Reduces Fine Lines and Wrinkles | Enjoy Glowing Skin

Semantic Results:


Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.34s/it]

  0.6193 | Greenwich Bay Hand Lotion For The Kitchen with Shea Butter, Cocoa Butter, Virgin Olive Oil and Orange, Grapefruit and Lemon extracts 16 oz (Sugar Lemon Citrus)
  0.6137 | Face 'n' Earth, LLC Sensitive Skin Care Cleanser Chamomile & Cucumber - Soothing, calming and gentle All Skins - Vegan 6.6oz
  0.6120 | Natural Face Wash for Sensitive Skin - Gentle Anti Aging Milk Facial Cleanser with Vitamin C and Vitamin E - 6.7 Ounces - Eve Hansen
  0.6115 | Senka Perfect emulsion Silky Moisture moisturizing lotion 150ml
  0.6082 | Kiss My Face, Moisture Shave, Lavender Shea, 3.4 fl oz (100 ml) - 2pc


In [29]:
results = hybrid_search(bm25, index, products, "moisturizer for sensitive skin", top_k=5, model=model)
for r in results:
    print(f"hybrid={r['hybrid_score']:.6f} | bm25_rank={r['bm25_rank']} | sem_rank={r['semantic_rank']} | {r['title']}")

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:22<00:00, 22.46s/it]


hybrid=0.031818 | bm25_rank=0 | sem_rank=6 | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pack of 3)
hybrid=0.030952 | bm25_rank=None | sem_rank=0 | Senka Perfect emulsion Silky Moisture moisturizing lotion 150ml
hybrid=0.030679 | bm25_rank=1 | sem_rank=None | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive Skin With Broad Spectrum Spf 15, 4 oz (Pack of 3)
hybrid=0.030679 | bm25_rank=None | sem_rank=1 | Living Nature Sensitive Skin Day Moisture Cream
hybrid=0.030415 | bm25_rank=None | sem_rank=2 | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For Daily Use With Dispensing Pump - With Aloe Vera Jojoba Oil Shea Butter Green Tea For Sensitive Dry And Oily Skin - Premium Nature


In [30]:
query = "moisturizer for sensitive skin"

print("BM25:")
for r in search_bm25(bm25, products, query, top_k=10):
    print(f"  {r['parent_asin']} | {r['title']}")

print("\nSemantic:")
for r in search_semantic(index, products, query, top_k=10, model=model):
    print(f"  {r['parent_asin']} | {r['title']}")

BM25:
  B00YF3J4OG | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pack of 3)
  B072BKHJ7Z | Aveeno Ultra-Calming Daily BldGO Moisturizer For Sensitive Skin With Broad Spectrum Spf 15, 4 oz (Pack of 3)
  B07F2P45NZ | Revitalizing Light-Weight Moisturizer SPF 15
  B0842CC6X9 | Simple Hydrating Light Moisturizer, 4.2 Ounce 6-pack
  B07B8PHZPM | BRTC Perfect Calming Cream 50ml, for Dry, Sensitive and Itchy Skin
  B07YHN22JY | Lesentia Vitamin C Serum - Anti Aging Face Moisturizer for Women & Men with Hyaluronic Acid, Rosehip Oil, Retinol, Turmeric, Coq10 for a Daily Moisturizer Great for Sensitive Skin Care (1fl oz)
  B096T29D7N | Codex Labs Skin Hydration Kit | Microbiome Friendly, Vegan, Calendula, Soothing for dry,flaky or itchy skin, Sensitive Skin, EWG
  B0B4JP5YD9 | ELLI K ESSENTIAL SINCERITY FROM AZ TIME REVERSE CREAM - Made in USA - Anti-Aging Face Moisturizer for Dry & Rough Skin – Repairing Treatment Cream – Highly Concentrated Formula, 1.76 oz.
  B07G8MHVX

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:28<00:00, 28.72s/it]


  B01JFPP3L6 | Senka Perfect emulsion Silky Moisture moisturizing lotion 150ml
  B014T3QBLK | Living Nature Sensitive Skin Day Moisture Cream
  B0757RW9XK | Daily Face Moisturizer Serum - Hydrating Vitamin Lotion For Daily Use With Dispensing Pump - With Aloe Vera Jojoba Oil Shea Butter Green Tea For Sensitive Dry And Oily Skin - Premium Nature
  B01IADYIM4 | Moisturel Therapeutic Lotion 14 oz (Pack of 3)
  B09PNWWBYX | Cetaphil Moisturizing Cream 20 oz, 2 Pack
  B07G8MHVXQ | Daily Body and Face Moisturizer by Majestic Pure - Wonder Moisturizing Lotion for Women and Men - Nourishes and Hydrates - for Dry and All Skin Types - 9 fl. oz.
  B00YF3J4OG | Simple Hydrating Light Moisturizer, for Sensitive Skin, 4.2 Ounce, (Pack of 3)
  B015TNTPVM | Professional Salon Hair Dryer with Touch Sensor,Negative Ionic Hair Blow Dryer,1600w 6 Speed and Heat Settings AC Motor Infrared Heat Low Noise Hair Dryer with Diffuser & Concentrator
  B01IADUTPY | Moisturel Therapeutic Lotion 14 oz (Pack of 9)
  

In [31]:
query = "vitamin c serum"

print("BM25:")
for r in search_bm25(bm25, products, query, top_k=10):
    print(f"  {r['parent_asin']} | {r['title']}")

print("\nSemantic:")
for r in search_semantic(index, products, query, top_k=10, model=model):
    print(f"  {r['parent_asin']} | {r['title']}")

BM25:
  B00HSX95A8 | Springs Organic Vitamin C Serum For Your Face - Vitamin C Serum|Organic Vitamin C + Vitamin E + Aloe Vera + Hyaluronic Acid Serum- Effective Strength 20% Vitamin C Serum with Vegan Hyaluronic Acid Leaves Your Skin Radiant & More Beautiful By Neutralizing Free Radicals. This Anti Aging Antiwrinkles Serum Will Give You The Results in few days or we refund You With No-Question-Asked. Your!
  B07NPX9LNS | Serum Sensation Vitamin C Serum with Hyaluronic Acid, Organic Anti-Aging, Brightening Serum and Acne Scar Treatment (Vitamin c 2 pack)
  B071177MV1 | Organic Vitamin C Serum for Face-Professional Strength-Organic Vitamin C Serum-Hyaluronic Acid-Vitamin E-Naturally Derived 20% Vitamin C The Best Vitamin C Serum (1 Ounce)
  B09987JTXF | NuOrganic Super C Serum 20% Vitamin C Serum for Face | Hyaluronic Acid, Plant-Based Stem Cells, Ferulic Acid, Vitamin E, MSM | Skin Brightening & Antioxidant-Rich Serum (15 ML)
  B0713SYBLM | Organic Vitamin C Serum for Face-Professional

Batches: 100%|████████████████████████████████████████████████████████████████████████████████| 1/1 [00:24<00:00, 24.45s/it]


  B01CM8G8PI | PURE VITAMIN C SERUM
  B016MNB4HQ | Vitamin C Serum for Face - C Booster With Hyaluronic Acid and Vitamin E - Anti Wrinkle - Anti Aging - Skin Repair For Age Spots and Sun Damage - For Men and Women
  B019D6WNJM | Vitamin C Serum - 10% Pure Vitamin C, Ascorbic Acid, Liposomal Technology, Antioxidant, Ultimate Collagen Booster, Brightener, Fade Dark Spots, Exfoliate, Even Tone/Texture (1 Oz / 30ml)
  B00S5LB08W | Upgraded 30% Vitamin C Serum with Hyaluronic Acid and Vit E,Anti Aging Face Serum for Face Eyes,Anti Wrinkle Vitamin C Facail Serum
  B07KGLLVGN | Vitamin C Serum for Face, Moisturizer SPF Cream & Eye Beauty Treatment TRIO w/Bonus Tote Bag Serious Skincare C3 Plasma Technology
  B000MXRFY4 | Vitamin C Plus Serum
  B07BT56PWQ | Liz K Super First C Serum Pure Vitamin C 13%
  B07NPX9LNS | Serum Sensation Vitamin C Serum with Hyaluronic Acid, Organic Anti-Aging, Brightening Serum and Acne Scar Treatment (Vitamin c 2 pack)
  B012HD1UG0 | Haba Special Care White Lady V